##  Setup and load the saved results

In [ ]:
!pip -q install timm scikit-learn
import os, json, shutil, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import roc_auc_score, roc_curve
from google.colab import drive
warnings.filterwarnings("ignore")

drive.mount("/content/drive")
DRIVE_DATA = "/content/drive/MyDrive/rfmid_256"
LOCAL_DATA = "/content/rfmid_256"
RESULTS = "/content/drive/MyDrive/rfmid_results"
OUT = os.path.join(RESULTS, "day4"); os.makedirs(OUT, exist_ok=True)

# only the test photos are needed here
os.makedirs(LOCAL_DATA, exist_ok=True)
shutil.copy(os.path.join(DRIVE_DATA, "dataset.csv"), LOCAL_DATA)
if not os.path.exists(os.path.join(LOCAL_DATA, "test")):
    shutil.copytree(os.path.join(DRIVE_DATA, "test"), os.path.join(LOCAL_DATA, "test"))

ds = pd.read_csv(os.path.join(LOCAL_DATA, "dataset.csv"))
cfg = json.load(open(os.path.join(RESULTS, "app_config.json")))
THR = cfg["threshold"]
pred_test = pd.read_csv(os.path.join(RESULTS, "vit_test_preds.csv"))
pred_val = pd.read_csv(os.path.join(RESULTS, "vit_val_preds.csv"))

test = ds[ds.split == "test"].reset_index(drop=True)
assert (test["ID"].values == pred_test["ID"].values).all(), "test predictions are not in the same order as dataset.csv"
test["prob"] = pred_test["prob"].values
test["y"] = pred_test["y"].values.astype(int)
test["pred"] = (test["prob"] >= THR).astype(int)
print("test photos:", len(test), "| threshold:", round(THR, 4))
print("Check: test AUC =", round(roc_auc_score(test["y"], test["prob"]), 4), "(Day 3 reported", round(cfg["test_metrics"]["auc"], 4), ")")

##  Helper functions (metrics with confidence intervals)

With small groups, a single number can mislead. A **95% confidence interval** (bootstrap: re-sample the group's photos 1,000 times) shows how uncertain each number is. A wide interval means "not enough photos to be sure".

In [ ]:
def point(y, p, thr):
    pred = p >= thr
    pos, neg = y == 1, y == 0
    sens = pred[pos].mean() if pos.any() else np.nan
    spec = (~pred[neg]).mean() if neg.any() else np.nan
    auc = roc_auc_score(y, p) if pos.any() and neg.any() else np.nan
    return auc, sens, spec

def boot_ci(y, p, thr, n_boot=1000, seed=0):
    rng = np.random.default_rng(seed); n = len(y); vals = []
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        vals.append(point(y[i], p[i], thr))
    vals = np.array(vals, dtype=float)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return np.nanpercentile(vals, 2.5, axis=0), np.nanpercentile(vals, 97.5, axis=0)

def group_table(df, col, thr=THR):
    rows = []
    for g, d in df.groupby(col, observed=True):
        y, p = d["y"].values, d["prob"].values
        auc, sens, spec = point(y, p, thr)
        lo, hi = boot_ci(y, p, thr)
        rows.append({"group": g, "n": len(d), "n_abnormal": int((y == 1).sum()), "n_healthy": int((y == 0).sum()),
                     "AUC": auc, "AUC_lo": lo[0], "AUC_hi": hi[0],
                     "sensitivity": sens, "sens_lo": lo[1], "sens_hi": hi[1],
                     "specificity": spec, "spec_lo": lo[2], "spec_hi": hi[2]})
    return pd.DataFrame(rows)

def plot_groups(tab, title, fname):
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    for a, (m, lo, hi) in zip(ax, [("AUC", "AUC_lo", "AUC_hi"), ("sensitivity", "sens_lo", "sens_hi"),
                                   ("specificity", "spec_lo", "spec_hi")]):
        v = tab[m].values.astype(float)
        err = np.vstack([v - tab[lo].values.astype(float), tab[hi].values.astype(float) - v])
        err = np.nan_to_num(err)
        a.bar(range(len(tab)), np.nan_to_num(v), yerr=err, capsize=5, color="#4c78a8")
        a.set_xticks(range(len(tab)))
        a.set_xticklabels([f"{g}\n(n={n})" for g, n in zip(tab["group"], tab["n"])], fontsize=9)
        a.set_ylim(0, 1.05); a.set_title(m); a.axhline(cfg["test_metrics"][{"AUC": "auc"}.get(m, m)], color="gray", ls="--", lw=1)
    fig.suptitle(title + "  (dashed line = overall test value; bars = 95% CI)")
    plt.tight_layout(); plt.savefig(os.path.join(OUT, fname), dpi=150); plt.show()
print("ready")

##  Bias analysis by image source (camera)

RFMiD was captured with **three different fundus cameras** (Pachade et al., 2021). The file does not say which camera took which photo, but the three **original image sizes** match three groups, so we use image size as a **proxy for the camera**. State this clearly in your report: it is an inference, not a recorded label.

In [ ]:
test["camera"] = test["orig_w"].astype(str) + "x" + test["orig_h"].astype(str)
print(test["camera"].value_counts().to_string())
tab_cam = group_table(test, "camera")
tab_cam.round(3).to_csv(os.path.join(OUT, "bias_by_camera.csv"), index=False)
display(tab_cam.round(3))
plot_groups(tab_cam, "ViT performance by image source (camera proxy)", "bias_by_camera.png")

##  Bias analysis by image quality (brightness and contrast)

Photos are split into three equal-sized groups (low / mid / high) using cut-points from the whole dataset. Darker or flatter photos are often harder.

In [ ]:
for col in ["brightness", "contrast"]:
    edges = ds[col].quantile([1 / 3, 2 / 3]).values
    test[col + "_grp"] = pd.cut(test[col], [-np.inf, edges[0], edges[1], np.inf], labels=["low", "mid", "high"])
    tab = group_table(test, col + "_grp")
    tab.round(3).to_csv(os.path.join(OUT, f"bias_by_{col}.csv"), index=False)
    display(tab.round(3))
    plot_groups(tab, f"ViT performance by {col}", f"bias_by_{col}.png")

##  Which diseases does the model miss?

For every disease with at least 5 test photos, the **detection rate** is the share of those photos the model flags as abnormal. We compare it with how many **training** photos the disease had: rare diseases usually have fewer examples to learn from.

In [ ]:
non_disease = {"ID", "Disease_Risk", "split", "path", "orig_w", "orig_h", "brightness", "contrast",
               "prob", "y", "pred", "camera", "brightness_grp", "contrast_grp"}
dis_cols = [c for c in ds.columns if c not in non_disease]
train_n = ds[ds.split == "train"][dis_cols].sum()
rows = []
for c in dis_cols:
    m = test[c] == 1
    if m.sum() >= 5:
        rows.append({"disease": c, "train_photos": int(train_n[c]), "test_photos": int(m.sum()),
                     "detection_rate": float(test.loc[m, "pred"].mean())})
dz = pd.DataFrame(rows).sort_values("detection_rate")
dz.round(3).to_csv(os.path.join(OUT, "detection_by_disease.csv"), index=False)
display(dz.round(3))

fig, ax = plt.subplots(1, 2, figsize=(15, max(4, 0.32 * len(dz) + 1)))
ax[0].barh([f"{d} (n={n})" for d, n in zip(dz["disease"], dz["test_photos"])], dz["detection_rate"], color="#4c78a8")
ax[0].set_xlim(0, 1.05); ax[0].set_xlabel("share flagged as abnormal"); ax[0].set_title("Detection rate per disease (test set)")
ax[1].scatter(dz["train_photos"], dz["detection_rate"], s=20 + 8 * dz["test_photos"], alpha=.6, color="#d62728")
for _, r in dz.iterrows():
    ax[1].annotate(r["disease"], (r["train_photos"], r["detection_rate"]), fontsize=8, xytext=(3, 3), textcoords="offset points")
ax[1].set_xscale("log"); ax[1].set_ylim(0, 1.05)
ax[1].set_xlabel("training photos with this disease (log scale)"); ax[1].set_ylabel("detection rate")
rho = dz["train_photos"].corr(dz["detection_rate"], method="spearman")
ax[1].set_title(f"Rarer diseases vs detection (Spearman rho = {rho:.2f})")
plt.tight_layout(); plt.savefig(os.path.join(OUT, "detection_by_disease.png"), dpi=150); plt.show()

# which diseases make up the missed (false negative) cases?
fn = test[(test.y == 1) & (test.pred == 0)]
miss = pd.DataFrame({"missed": fn[dis_cols].sum(), "all_in_test": test[dis_cols].sum()})
miss = miss[miss["missed"] > 0].sort_values("missed", ascending=False)
miss["miss_rate"] = miss["missed"] / miss["all_in_test"]
print(f"Missed abnormal photos (false negatives): {len(fn)} of {int((test.y == 1).sum())}")
display(miss.head(12).round(3))
miss.round(3).to_csv(os.path.join(OUT, "missed_by_disease.csv"))

In [ ]:
ab = test[test.y == 1].copy()
def rarest_group(row):
    present = [c for c in dis_cols if row[c] == 1]
    if not present: return "no specific disease labelled"
    n = min(train_n[c] for c in present)
    return "rare (<20 train photos)" if n < 20 else ("moderate (20-99)" if n < 100 else "common (100+)")
ab["freq_group"] = ab.apply(rarest_group, axis=1)
rows = []
for g, d in ab.groupby("freq_group"):
    rate = d["pred"].mean()
    rng = np.random.default_rng(0)
    boots = [d["pred"].values[rng.integers(0, len(d), len(d))].mean() for _ in range(1000)]
    rows.append({"group": g, "n_photos": len(d), "detection_rate": rate,
                 "lo": np.percentile(boots, 2.5), "hi": np.percentile(boots, 97.5)})
fq = pd.DataFrame(rows).sort_values("detection_rate")
fq.round(3).to_csv(os.path.join(OUT, "detection_by_frequency_group.csv"), index=False)
display(fq.round(3))
plt.figure(figsize=(7, 3.6))
plt.bar([f"{g}\n(n={n})" for g, n in zip(fq["group"], fq["n_photos"])], fq["detection_rate"],
        yerr=[fq["detection_rate"] - fq["lo"], fq["hi"] - fq["detection_rate"]], capsize=5, color="#4c78a8")
plt.ylim(0, 1.05); plt.ylabel("detection rate (sensitivity)"); plt.title("Detection by disease frequency (95% CI)")
plt.tight_layout(); plt.savefig(os.path.join(OUT, "detection_by_frequency_group.png"), dpi=150); plt.show()

## Choosing a safer threshold

For a screening tool, **missing a sick eye is worse than sending a healthy person for an extra check**. The threshold so far maximises sensitivity + specificity equally. Here we pick, on the **validation** set, the threshold that reaches at least 95% sensitivity, then report what that does on the **test** set. (The test set is never used to choose the threshold.)

In [ ]:
yv, pv = pred_val["y"].values.astype(int), pred_val["prob"].values
fpr, tpr, thr = roc_curve(yv, pv)
i95 = int(np.argmax(tpr >= 0.95))
THR95 = float(thr[i95])

def at(thr_):
    a, s, sp = point(test["y"].values, test["prob"].values, thr_)
    pred = test["prob"].values >= thr_
    return {"threshold": round(thr_, 4), "sensitivity": round(s, 3), "specificity": round(sp, 3),
            "missed_abnormal": int(((test.y == 1) & (~pred)).sum()), "healthy_flagged": int(((test.y == 0) & pred).sum())}
op = pd.DataFrame([{"setting": "balanced (Day 3)", **at(THR)}, {"setting": "high sensitivity (>=95% on val)", **at(THR95)}])
display(op)
op.to_csv(os.path.join(OUT, "operating_points.csv"), index=False)

ts = np.linspace(0.01, 0.99, 197)
sens_ = [point(test["y"].values, test["prob"].values, t)[1] for t in ts]
spec_ = [point(test["y"].values, test["prob"].values, t)[2] for t in ts]
plt.figure(figsize=(7, 4))
plt.plot(ts, sens_, label="sensitivity", color="#d62728"); plt.plot(ts, spec_, label="specificity", color="#2ca02c")
plt.axvline(THR, color="k", ls="--", label=f"balanced threshold ({THR:.2f})")
plt.axvline(THR95, color="gray", ls=":", label=f"high-sensitivity threshold ({THR95:.2f})")
plt.xlabel("decision threshold"); plt.ylabel("test-set value"); plt.legend(); plt.title("Sensitivity and specificity versus threshold (test set)")
plt.tight_layout(); plt.savefig(os.path.join(OUT, "threshold_tradeoff.png"), dpi=150); plt.show()

## Attention heatmaps



In [ ]:
import torch, timm
import torchvision.transforms as T

device = "cuda" if torch.cuda.is_available() else "cpu"
model = timm.create_model(cfg["timm_name"], pretrained=False, num_classes=1)
model.load_state_dict(torch.load(os.path.join(RESULTS, "vit_best.pt"), map_location="cpu"))
model.eval().to(device)

attn_maps = []
def hook(module, inp, out): attn_maps.append(out.detach().cpu())
for blk in model.blocks:
    blk.attn.fused_attn = False            # make the attention matrix available
    blk.attn.attn_drop.register_forward_hook(hook)

tf = T.Compose([T.Resize((cfg["img_size"], cfg["img_size"])), T.ToTensor(), T.Normalize(cfg["mean"], cfg["std"])])

def rollout(path):
    img = Image.open(path).convert("RGB")
    x = tf(img).unsqueeze(0).to(device)
    attn_maps.clear()
    with torch.no_grad():
        prob = torch.sigmoid(model(x)).item()
    result = torch.eye(attn_maps[0].shape[-1])
    for a in attn_maps:                       # each: (1, heads, tokens, tokens)
        a = a.mean(dim=1)[0]
        a = a + torch.eye(a.shape[-1])
        a = a / a.sum(dim=-1, keepdim=True)
        result = a @ result
    mask = result[0, 1:]                      # class token -> image patches
    side = int(round(mask.numel() ** 0.5))
    mask = mask.reshape(side, side).numpy()
    mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)
    heat = np.asarray(Image.fromarray((mask * 255).astype("uint8")).resize((cfg["img_size"],) * 2, Image.BICUBIC)) / 255.0
    return np.asarray(img.resize((cfg["img_size"],) * 2)), heat, prob

groups = {"True positive (abnormal, flagged)": test[(test.y == 1) & (test.pred == 1)],
          "True negative (healthy, cleared)": test[(test.y == 0) & (test.pred == 0)],
          "False negative (abnormal, MISSED)": test[(test.y == 1) & (test.pred == 0)],
          "False positive (healthy, flagged)": test[(test.y == 0) & (test.pred == 1)]}
for k, (title, d) in enumerate(groups.items()):
    d = d.sample(n=min(4, len(d)), random_state=0)
    fig, ax = plt.subplots(2, len(d), figsize=(3.2 * len(d), 6.6), squeeze=False)
    for j, (_, r) in enumerate(d.iterrows()):
        img, heat, prob = rollout(os.path.join(LOCAL_DATA, r["path"]))
        ax[0, j].imshow(img); ax[0, j].set_title(f"photo {r['ID']}  P(abnormal)={prob:.2f}", fontsize=9)
        ax[1, j].imshow(img); ax[1, j].imshow(heat, cmap="jet", alpha=0.45)
        for a in (ax[0, j], ax[1, j]): a.axis("off")
    fig.suptitle(title + f"   [{len(groups[title])} such photos in the test set]")
    plt.tight_layout(); plt.savefig(os.path.join(OUT, f"attention_{k + 1}.png"), dpi=130); plt.show()